# Customer Churn Prediction System
## Telco Customer Churn – Full ML Pipeline
### ChurnIQ | YHILLS Internship Project

---

**Dataset:** IBM Telco Customer Churn (~7043 records, 20+ features)  
**Target:** Churn (Yes / No)  
**Models:** Logistic Regression · Decision Tree · Random Forest · SVM · Gradient Boosting · XGBoost  
**Best Metric:** ROC AUC with Stratified K-Fold cross-validation


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection  import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm              import SVC
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from xgboost import XGBClassifier
import joblib, os, sys

sys.path.insert(0, '..')

# Dark plot theme
plt.rcParams.update({
    'figure.facecolor': '#0F172A', 'axes.facecolor': '#0F172A',
    'axes.edgecolor':   '#334155', 'axes.labelcolor': '#E2E8F0',
    'xtick.color':      '#E2E8F0', 'ytick.color':     '#E2E8F0',
    'text.color':       '#E2E8F0', 'grid.color':      '#334155',
    'legend.facecolor': '#1E293B',
})
print('Libraries imported successfully ✅')

## 2. Load Dataset

In [ ]:
from src.data_loader import load_data, get_basic_info

df = load_data('../dataset/telco_churn.csv')
info = get_basic_info(df)

print('Dataset Info:')
for k, v in info.items():
    print(f'  {k}: {v}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ── Churn Distribution ───────────────────────────────────────────
churn_counts = df['Churn'].value_counts()
print('Churn Distribution:')
print(churn_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#10B981', '#EF4444'], edgecolor='#0F172A')
axes[0].set_title('Churn Count', color='white', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Customers', color='white')

axes[1].pie(churn_counts.values, labels=churn_counts.index,
            colors=['#10B981', '#EF4444'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Rate', color='white', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Churn by Contract Type ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
pd.crosstab(df['Contract'], df['Churn'], normalize='index').mul(100).plot(
    kind='bar', ax=ax, color=['#10B981', '#EF4444'], edgecolor='#0F172A'
)
ax.set_title('Churn Rate by Contract Type (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type'); ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Monthly Charges vs Churn ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[df['Churn']=='No']['MonthlyCharges'],  bins=30, alpha=0.7, color='#10B981', label='No Churn')
ax.hist(df[df['Churn']=='Yes']['MonthlyCharges'], bins=30, alpha=0.7, color='#EF4444', label='Churn')
ax.set_title('Monthly Charges by Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)'); ax.set_ylabel('Count')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Tenure Distribution ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[df['Churn']=='No']['tenure'],  bins=30, alpha=0.7, color='#10B981', label='No Churn')
ax.hist(df[df['Churn']=='Yes']['tenure'], bins=30, alpha=0.7, color='#EF4444', label='Churn')
ax.set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)'); ax.set_ylabel('Count')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Data Preprocessing

In [ ]:
from src.preprocessing import clean_data, encode_categoricals

df_clean = clean_data(df.copy())
print('After cleaning:', df_clean.shape)
print('Missing values:', df_clean.isnull().sum().sum())
print('Churn distribution:')
print(df_clean['Churn'].value_counts())
df_clean.describe()

## 5. Feature Engineering

In [ ]:
from src.feature_engineering import add_service_count, add_arpu, add_high_risk_flags

df_eng = add_service_count(df_clean.copy())
df_eng = add_arpu(df_eng)
df_eng = add_high_risk_flags(df_eng)

print('New engineered features:')
print(df_eng[['tenure', 'MonthlyCharges', 'TotalCharges', 'service_count', 'ARPU', 'high_monthly']].describe())

In [ ]:
# Encode all categoricals
df_encoded = encode_categoricals(df_eng.copy())
print('After OHE encoding:', df_encoded.shape)

TARGET = 'Churn'
y = df_encoded[TARGET].astype(int)
X = df_encoded.drop(columns=[TARGET])
print('X shape:', X.shape, '| y churn rate:', f'{y.mean()*100:.1f}%')
print('\nFeature list:')
print(list(X.columns))

## 6. Train / Test Split & Scaling

In [ ]:
from src.config import NUMERIC_FEATURES

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = [c for c in NUMERIC_FEATURES if c in X_train.columns]
scaler   = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean()*100:.1f}%  |  Test: {y_test.mean()*100:.1f}%')

## 7. Model Training with Cross-Validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVM':                 SVC(probability=True, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Cross-Validation ROC AUC Scores (5-Fold):')
print('-' * 50)
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'  {name:<25}  {scores.mean():.4f}  ±  {scores.std():.4f}')

print('\nFitting on full training set...')
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'  ✅ {name}')

## 8. Model Comparison

In [ ]:
results = []
for name, model in models.items():
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1 Score':  round(f1_score(y_test, y_pred), 4),
        'ROC AUC':   round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results).sort_values('ROC AUC', ascending=False).reset_index(drop=True)
results_df.index += 1
print('Model Comparison (sorted by ROC AUC):')
results_df

In [ ]:
# ── Model Comparison Bar Chart ───────────────────────────────────────────────
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
x = np.arange(len(results_df['Model']))
width = 0.15
colours = ['#6366F1', '#F59E0B', '#10B981', '#EF4444', '#3B82F6']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, colour) in enumerate(zip(metrics_to_plot, colours)):
    ax.bar(x + (i - 2) * width, results_df[metric], width, label=metric, color=colour, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 9. Evaluation Metrics – Best Model

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
print(f'Best Model: {best_name}')

y_pred_best  = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churn']))

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colours = ['#6366F1', '#F59E0B', '#10B981', '#EF4444', '#3B82F6', '#EC4899']

for (name, model), colour in zip(models.items(), colours):
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    ax1.plot(fpr, tpr, lw=2, color=colour, label=f'{name} (AUC={auc:.3f})')

ax1.plot([0,1],[0,1], 'w--', lw=1, label='Random')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curves – All Models', fontweight='bold')
ax1.legend(loc='lower right', fontsize=7)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'], ax=ax2)
ax2.set_title(f'Confusion Matrix – {best_name}', fontweight='bold')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')

plt.tight_layout(); plt.show()

In [ ]:
# ── Feature Importance ──────────────────────────────────────────────────────
feature_names = X.columns.tolist()

if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=feature_names).nlargest(20).sort_values()
    fig, ax = plt.subplots(figsize=(10, 7))
    fi.plot(kind='barh', color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)), ax=ax, edgecolor='#0F172A')
    ax.set_title(f'Top 20 Feature Importances – {best_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout(); plt.show()
elif hasattr(best_model, 'coef_'):
    fi = pd.Series(np.abs(best_model.coef_[0]), index=feature_names).nlargest(20).sort_values()
    fi.plot(kind='barh', color='#6366F1', figsize=(10, 7))
    plt.title(f'Coefficient Importance – {best_name}', fontweight='bold')
    plt.tight_layout(); plt.show()

## 10. Save Trained Model

In [ ]:
os.makedirs('../models', exist_ok=True)

payload = {
    'model':         best_model,
    'scaler':        scaler,
    'feature_names': feature_names,
    'model_name':    best_name,
    'metrics':       results_df.iloc[0].to_dict(),
    'all_results':   results,
}

joblib.dump(payload, '../models/churn_model.pkl')

print('=' * 50)
print(' MODEL SAVED SUCCESSFULLY')
print('=' * 50)
print(f'  Best Model:  {best_name}')
print(f'  Accuracy:    {results_df.iloc[0]["Accuracy"]*100:.1f}%')
print(f'  ROC AUC:     {results_df.iloc[0]["ROC AUC"]*100:.1f}%')
print(f'  F1 Score:    {results_df.iloc[0]["F1 Score"]*100:.1f}%')
print(f'  Saved to:    ../models/churn_model.pkl')